# Pre-class: Monday Morning — Seeing the Seasonality
**⏱ This pre-class notebook takes approximately 15 minutes.**

---

## Scenario: Monday — Marcus's Forecasting Brief

It's week 6 at NorthStar Retail. Friday last week, Marcus said:

> *"Now — our sales are SEASONAL. We need to know quarterly revenue 90 days ahead so ops can plan inventory. Can you forecast?"*

This Monday Sarah opens `northstar_daily_revenue.csv` — 731 days of daily revenue, January 2024 through December 2025. Her job by Friday:

1. Decompose the series into trend + seasonality + residual.
2. Train and compare multiple forecasting methods.
3. Produce a 90-day forecast with honest error bars.

**By the end of this notebook you will be able to:**
- Open a time-series CSV and put it on a date index
- See the trend, weekly seasonality, and annual seasonality with the naked eye
- Understand why this data needs a different toolkit than L03/L04 supervised learning

In [ ]:
# Setup
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings

warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (13, 4.5)

print("✅ Libraries loaded")

## Load the data and parse dates

Time series ALWAYS work best when the DataFrame index is a `DatetimeIndex`. We parse `date` as a date column and set it as the index.

In [ ]:
df = pd.read_csv("data/northstar_daily_revenue.csv", parse_dates=["date"])
df = df.set_index("date")

print(f"Rows: {len(df):,}")
print(f"Date range: {df.index.min().date()} → {df.index.max().date()}")
print(f"Columns: {list(df.columns)}")
print()
df.head()

## Step 1 — The whole series at a glance

Plot the daily revenue. What does it look like?

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df.index, df["revenue_gbp"], linewidth=0.8, color="steelblue")
ax.set_xlabel("Date")
ax.set_ylabel("Daily revenue (£)")
ax.set_title("NorthStar daily revenue — 2024-01-01 to 2025-12-31")
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print(f"Mean daily revenue: £{df['revenue_gbp'].mean():,.0f}")
print(f"Min: £{df['revenue_gbp'].min():,.0f}  |  Max: £{df['revenue_gbp'].max():,.0f}")

### 💡 What you should notice

Three patterns visible in the plot:

1. **A clear upward TREND** — revenue is generally higher in 2025 than 2024
2. **An ANNUAL spike around November-December** — the holiday-shopping bump
3. **A high-frequency oscillation** that's hard to see at this zoom level — probably weekly

This is a multi-seasonal series. Let's zoom in to see the weekly pattern.

## Step 2 — Zoom in on a few weeks

In [ ]:
# First two months
sample = df.loc["2024-01-01":"2024-02-29"]

fig, ax = plt.subplots(figsize=(14, 4.5))
ax.plot(sample.index, sample["revenue_gbp"], "o-", markersize=6, linewidth=1.2, color="steelblue")
ax.set_xlabel("Date")
ax.set_ylabel("Daily revenue (£)")
ax.set_title("Two months zoom — see the weekly oscillation")
ax.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=0))  # Mondays
ax.xaxis.set_major_formatter(mdates.DateFormatter("%a %d"))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Mean by day of week
print("Mean revenue by day of week:")
df["day_name"] = df.index.day_name()
print(df.groupby("day_name", sort=False)["revenue_gbp"].mean().round(0).to_string())

### 💡 What you should notice

- **Weekends (Sat, Sun) consistently higher** than weekdays.
- **Friday is a moderate lift**, weekdays are flat.
- This is the **weekly seasonality** we couldn't see in the full-2-year zoom.

Two seasonalities = annual (visible at full zoom) AND weekly (visible at month zoom). The forecasting methods will need to handle both.

## Step 3 — Compare the two years to see the annual pattern

In [ ]:
# Add year and dayofyear; overlay 2024 vs 2025
df["year"]       = df.index.year
df["dayofyear"]  = df.index.dayofyear

fig, ax = plt.subplots(figsize=(13, 5))
for year, color in [(2024, "steelblue"), (2025, "coral")]:
    yr_df = df[df["year"] == year]
    ax.plot(yr_df["dayofyear"], yr_df["revenue_gbp"],
            alpha=0.7, linewidth=0.8, label=f"{year}", color=color)
ax.set_xlabel("Day of year")
ax.set_ylabel("Daily revenue (£)")
ax.set_title("Annual cycle — 2024 vs 2025 overlaid")
ax.axvspan(320, 360, alpha=0.15, color="orange", label="Nov-Dec holiday window")
ax.legend()
plt.tight_layout()
plt.show()

### 💡 What you should notice

- **2025 is consistently higher than 2024** — that's the trend.
- **Both years spike around days 320-360** — late November through December. Holiday shopping.
- The annual pattern is **stable** — it repeats. That makes it forecastable.

By Friday, Sarah's forecast will:
1. Project the trend forward
2. Add the annual seasonal lift (peak in Q4)
3. Add the weekly cycle (weekend boost)
4. Account for residual uncertainty

## Step 4 — A simple eyeball forecast

Before any model, what would YOU predict? For example: revenue on April 1, 2026 (about 90 days after the data ends).

April 1 is day-of-year 91. Look at the same date in 2024 and 2025.

In [ ]:
apr1_2024 = df.loc["2024-04-01", "revenue_gbp"]
apr1_2025 = df.loc["2025-04-01", "revenue_gbp"]

# Naive trend: assume year-over-year growth continues
yoy_growth = apr1_2025 - apr1_2024
apr1_2026_eyeball = apr1_2025 + yoy_growth

print(f"April 1, 2024:  £{apr1_2024:,.0f}")
print(f"April 1, 2025:  £{apr1_2025:,.0f}")
print(f"Year-over-year change: £{yoy_growth:+,.0f}")
print()
print(f"Eyeball forecast for April 1, 2026:  £{apr1_2026_eyeball:,.0f}")
print()
print("(This is the 'just project the trend' method. Real forecasting can do better — and quantify uncertainty.)")

## ✅ Section Summary

| What we did | What we saw |
|---|---|
| **Plotted the full 2-year series** | Upward trend + Q4 holiday spike |
| **Zoomed to a 2-month window** | Weekly seasonality (weekends > weekdays) |
| **Overlaid 2024 vs 2025** | Same annual shape, higher overall in 2025 |
| **Eyeball forecast** | Year-over-year change projects the trend, but no uncertainty estimate |

**Bring to class:**
1. The overall trend direction (growth, decline, or flat).
2. The dominant seasonal patterns you can name.
3. Your eyeball forecast for April 1, 2026 — we'll see how the models compare.

---
**In class → Open `02_decomposition.ipynb` first.** That notebook formalises the trend/seasonality/residual decomposition with STL.